# Contribuição C, passos 1 e 2: inspeção dos corpora, vazamento de treino e lista de termos

Este notebook:
1. carrega os corpora externos que estão em `data_externa` e mostra tamanho, rótulos e licença de cada um;
2. mede quantos textos de cada corpus aparecem nos dados de treino dos dois classificadores de toxicidade;
3. monta a tabela de vazamento (modelo por corpus);
4. grava a lista congelada de termos de deficiência;
5. conta os textos por termo e por rótulo em cada corpus.

**Entrada:** pasta `data_externa` (gerada pelo notebook 01).
**Saída:** arquivos `c1_...` e `c2_...` na pasta `results`, e `lista_termos_congelada.csv` na pasta raiz.

Modelos: BERTimbau toxic (`dougtrajano/toxic-comment-classification`, ajustado no OLID-BR) e ToxiGuardrail-PT (`nicholasKluge/ToxiGuardrailPT`, treinado no `nicholasKluge/harmful-text`). O BERTweet-PT foi ajustado no `pysentimiento/pt_sentiment`, que é fechado, então não entra nas comparações de texto.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import re, unicodedata
import numpy as np, pandas as pd

ROOT = "/content/drive/MyDrive/JIS - capacitismo"
BASE = f"{ROOT}/data_externa"
OUT  = f"{ROOT}/results"

def norm(s):
    s = str(s).lower()
    s = re.sub(r'https?://\S+|www\.\S+', ' ', s)
    s = re.sub(r'@\w+', ' ', s)
    s = re.sub(r'^rt\b', ' ', s)
    s = re.sub(r'\buser\b', ' ', s)
    s = unicodedata.normalize('NFKC', s)
    s = re.sub(r'[^\w\s]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

def rp(name):
    return pd.read_parquet(f"{BASE}/{name}.parquet")

ht = rp("harmful_text_pt_train")
harmful_text = pd.concat([pd.DataFrame({"text": ht.harmless, "label": "harmless"}),
                          pd.DataFrame({"text": ht.harmful, "label": "harmful"})], ignore_index=True)

olid = pd.concat([rp("olid_br_train").assign(split="train"), rp("olid_br_test").assign(split="test")], ignore_index=True)
olid["label"] = olid.is_offensive.map({"OFF": "ofensivo", "NOT": "não ofensivo"})

hatebr = pd.read_csv(f"{BASE}/hatebr.csv").rename(columns={"comentario": "text"})
hatebr["label"] = hatebr.label_final.map({1: "ofensivo", 0: "não ofensivo"})

told = pd.concat([pd.read_csv(f"{BASE}/told_br/1annotator/ptbr_{s}_1annotator.csv").assign(split=s)
                  for s in ["train", "validation", "test"]], ignore_index=True)
told["label"] = told.toxic.map({1: "tóxico", 0: "não tóxico"})

toxsyn = pd.concat([rp("toxsyn_pt_train").assign(split="train"), rp("toxsyn_pt_test").assign(split="test")], ignore_index=True)
toxsyn["label"] = toxsyn.is_toxic.map({1: "tóxico", 0: "não tóxico"})

hatecheck = rp("hatecheck_pt_test").rename(columns={"test_case": "text"})
hatecheck["label"] = hatecheck.label_gold.map({"hateful": "ódio", "non-hateful": "não ódio"})

tupye = pd.concat([rp("tupye_binary_train").assign(split="train"), rp("tupye_binary_test").assign(split="test")], ignore_index=True)
tupye["label"] = tupye.hate.map({1: "ódio", 0: "não ódio"})

CORPORA = {"harmful-text (PT)": harmful_text, "OLID-BR": olid, "HateBR": hatebr, "ToLD-Br": told,
           "ToxSyn-PT": toxsyn, "HateCheck-PT": hatecheck, "TuPy-E": tupye}
for d in CORPORA.values():
    d["text"] = d.text.astype(str)
    d["norm"] = d.text.map(norm)
print("corpora carregados:", {k: len(v) for k, v in CORPORA.items()})

Mounted at /content/drive
corpora carregados: {'harmful-text (PT)': 56206, 'OLID-BR': 6952, 'HateBR': 7000, 'ToLD-Br': 21000, 'ToxSyn-PT': 53274, 'HateCheck-PT': 3691, 'TuPy-E': 43668}


## 1. Inspeção dos corpora

Tamanho, número de textos distintos (depois de normalizar), distribuição de rótulos, tamanho mediano e licença. A licença vem da ficha do Hugging Face ou do arquivo LICENSE do repositório. A do ToxSyn-PT não está declarada na ficha.

In [2]:
LICENCAS = {"harmful-text (PT)": "Apache 2.0 (ficha)", "OLID-BR": "CC BY 4.0 (ficha)",
            "HateBR": "CC BY-NC 4.0 (arquivo LICENSE)", "ToLD-Br": "CC BY-SA 4.0 (arquivo LICENSE)",
            "ToxSyn-PT": "não declarada na ficha", "HateCheck-PT": "CC BY 4.0 (ficha)", "TuPy-E": "CC BY 4.0 (ficha)"}

inspecao = pd.DataFrame([{
    "corpus": name, "textos": len(d),
    "textos_distintos": d.norm.nunique(),
    "rotulos": d.label.value_counts().to_dict(),
    "mediana_palavras": int(d.text.str.split().str.len().median()),
    "licenca": LICENCAS[name]} for name, d in CORPORA.items()])
inspecao.to_csv(f"{OUT}/c1_inspecao_corpora.csv", index=False)
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 80)
print(inspecao.to_string(index=False))

print("\nTuPy-E por origem:")
print(tupye.groupby(["researcher", "source", "year"]).size())
tupye_ml = pd.concat([rp("tupye_multilabel_train"), rp("tupye_multilabel_test")], ignore_index=True)
print("\nTuPy-E, textos marcados como capacitismo:", int(tupye_ml.capacitism.sum()), "de", len(tupye_ml),
      "| por origem:", tupye_ml.groupby("researcher").capacitism.sum().to_dict())

           corpus  textos  textos_distintos                                  rotulos  mediana_palavras                        licenca
harmful-text (PT)   56206             55889    {'harmless': 28103, 'harmful': 28103}                34             Apache 2.0 (ficha)
          OLID-BR    6952              6934 {'ofensivo': 5936, 'não ofensivo': 1016}                16              CC BY 4.0 (ficha)
           HateBR    7000              6996 {'ofensivo': 3500, 'não ofensivo': 3500}                 9 CC BY-NC 4.0 (arquivo LICENSE)
          ToLD-Br   21000             20567    {'não tóxico': 11745, 'tóxico': 9255}                12 CC BY-SA 4.0 (arquivo LICENSE)
        ToxSyn-PT   53274             53250   {'não tóxico': 28496, 'tóxico': 24778}                26         não declarada na ficha
     HateCheck-PT    3691              3691         {'ódio': 2581, 'não ódio': 1110}                 8              CC BY 4.0 (ficha)
           TuPy-E   43668             42223        {'não ódio'

## 2. Sobreposição com os dados de treino

Para cada corpus (ou partição), conta quantos textos aparecem no `harmful-text` (treino do ToxiGuardrail-PT) e no OLID-BR (ajuste do BERTimbau toxic).

A comparação usa o texto normalizado: minúsculas, sem links, menções e pontuação. Isso é necessário porque o `harmful-text` remove menções e pontuação dos tuítes.

A coluna "com mais de 20 caracteres" desconta frases curtas e genéricas (por exemplo, "bando de canalhas"), que coincidem por acaso em qualquer corpus.

In [3]:
TREINO = {"harmful-text (treino do ToxiGuardrail-PT)": set(harmful_text.norm),
          "OLID-BR (ajuste do BERTimbau toxic)": set(olid.norm)}

PARTES = {"HateBR": hatebr,
          "ToLD-Br treino": told[told.split == "train"],
          "ToLD-Br validação": told[told.split == "validation"],
          "ToLD-Br teste": told[told.split == "test"],
          "HateCheck-PT": hatecheck,
          "ToxSyn-PT treino": toxsyn[toxsyn.split == "train"],
          "ToxSyn-PT teste": toxsyn[toxsyn.split == "test"],
          "OLID-BR": olid,
          "harmful-text (PT)": harmful_text}
for origem in sorted(tupye.researcher.unique()):
    PARTES[f"TuPy-E ({origem})"] = tupye[tupye.researcher == origem]

rows = []
for parte, d in PARTES.items():
    row = {"corpus": parte, "textos": len(d)}
    for treino, conjunto in TREINO.items():
        achou = d.norm.isin(conjunto)
        row[f"em {treino}"] = int(achou.sum())
        row[f"em {treino}, com mais de 20 caracteres"] = int((achou & (d.norm.str.len() > 20)).sum())
    rows.append(row)
sobreposicao = pd.DataFrame(rows)
sobreposicao.to_csv(f"{OUT}/c2_sobreposicao_corpora.csv", index=False)
print(sobreposicao.to_string(index=False))

                 corpus  textos  em harmful-text (treino do ToxiGuardrail-PT)  em harmful-text (treino do ToxiGuardrail-PT), com mais de 20 caracteres  em OLID-BR (ajuste do BERTimbau toxic)  em OLID-BR (ajuste do BERTimbau toxic), com mais de 20 caracteres
                 HateBR    7000                                             1                                                                        0                                       3                                                                  1
         ToLD-Br treino   16800                                         11774                                                                    10328                                     333                                                                280
      ToLD-Br validação    2100                                          1459                                                                     1254                                      40                                    

## 3. Tabela de vazamento

Regra usada para preencher cada célula, a partir da contagem do passo 2:
- nenhum texto coincidente: "não visto";
- menos de 3% dos textos: "não visto, salvo N textos coincidentes";
- mais de 50%: "visto";
- entre 3% e 50%: "parcialmente visto".

Os dados de treino do BERTweet-PT não estão disponíveis, então a coluna dele fica como "não verificado".

In [4]:
def status(n, total, proprio=False):
    if proprio:
        return "visto (é o dado de treino, segundo a ficha)"
    frac = n / total
    if n == 0:
        return "não visto (0 textos coincidentes)"
    if frac < 0.03:
        return f"não visto, salvo {n} texto(s) coincidente(s) ({frac:.1%})"
    if frac > 0.5:
        return f"visto ({n} de {total} textos, {frac:.0%})"
    return f"parcialmente visto ({n} de {total} textos, {frac:.0%})"

col_tox2, col_tox1 = list(TREINO)
vaz = []
for _, r in sobreposicao.iterrows():
    vaz.append({"corpus": r.corpus,
                "BERTimbau toxic": status(r[f"em {col_tox1}"], r.textos, proprio=(r.corpus == "OLID-BR")),
                "ToxiGuardrail-PT": status(r[f"em {col_tox2}"], r.textos, proprio=(r.corpus == "harmful-text (PT)")),
                "BERTweet-PT": "não verificado (dados de treino fechados)"})
vaz.append({"corpus": "TweetSentBR (pysentimiento/pt_sentiment)",
            "BERTimbau toxic": "não verificado (textos indisponíveis)",
            "ToxiGuardrail-PT": "não verificado (textos indisponíveis)",
            "BERTweet-PT": "visto (é o dado de treino, segundo o código do pysentimiento)"})
vazamento = pd.DataFrame(vaz)
vazamento.to_csv(f"{OUT}/c2_tabela_vazamento.csv", index=False)
pd.set_option("display.max_colwidth", None)
print(vazamento.to_string(index=False))

                                  corpus                                     BERTimbau toxic                                   ToxiGuardrail-PT                                                   BERTweet-PT
                                  HateBR   não visto, salvo 3 texto(s) coincidente(s) (0.0%)  não visto, salvo 1 texto(s) coincidente(s) (0.0%)                     não verificado (dados de treino fechados)
                          ToLD-Br treino não visto, salvo 333 texto(s) coincidente(s) (2.0%)                 visto (11774 de 16800 textos, 70%)                     não verificado (dados de treino fechados)
                       ToLD-Br validação  não visto, salvo 40 texto(s) coincidente(s) (1.9%)                   visto (1459 de 2100 textos, 69%)                     não verificado (dados de treino fechados)
                           ToLD-Br teste  não visto, salvo 41 texto(s) coincidente(s) (2.0%) não visto, salvo 53 texto(s) coincidente(s) (2.5%)                     não verifica

## 4. Lista congelada de termos

Cada linha é um termo de deficiência. Colunas:
- `termo`: nome que vai na coluna `SubClass` da v2;
- `subclass_v1`: nome do mesmo termo na v1 (para juntar com as tabelas da v1.1);
- `grupo`: Clínico, Discurso Social ou "Só E" (xingamentos usados só na contribuição E, fora do corpus);
- `padrao_busca`: expressão regular aplicada ao texto em minúsculas e sem acentos;
- `entra_no_v2` e `subconjunto_metaforico`.

Decisões registradas:
- formas derivadas ("cegueira", "surdez") ficam fora da contagem, porque o BITS testa "cego" e "surdo";
- "autista", "deficiente" e "deficiente mental" são acréscimos do plano (item i) e foram depois incluídos na v2;
- os xingamentos são "retardado" e "mongol".

Limitações da busca: ela não distingue sentidos ("ponto cego", "governo deficiente") e os termos não são excludentes ("deficiente físico" também conta em "deficiente").

In [7]:
DATA_CONGELAMENTO = "2026-10-12"

TERMOS = [
    ("com transtorno do espectro autista",   "com Transtorno do Espectro Autista",   "Clínico", r"transtorno do espectro autista", "sim", "não"),
    ("com transtorno de déficit de atenção", "com Transtorno de Déficit de Atenção", "Clínico", r"deficit de atencao|\btdah\b", "sim", "não"),
    ("com paralisia cerebral",               "com Paralisia Cerebral",               "Clínico", r"paralisia cerebral", "sim", "não"),
    ("com deficiência visual",               "com Deficiência Visual",               "Clínico", r"deficiencia visual", "sim", "não"),
    ("com perda auditiva",                   "com Perda Auditiva",                   "Clínico", r"perda auditiva|deficiencia auditiva", "sim", "não"),
    ("com síndrome de down",                 "com Síndrome de Down",                 "Clínico", r"sindrome de down", "sim", "não"),
    ("cego",                    "cego",                    "Discurso Social", r"\bceg[oa]s?\b", "sim", "literal e figurado"),
    ("surdo",                   "surdo",                   "Discurso Social", r"\bsurd[oa]s?\b", "sim", "literal e figurado"),
    ("cadeirante",              "cadeirante",              "Discurso Social", r"\bcadeirantes?\b", "sim", "só literal"),
    ("pessoa com deficiência",  "pessoa com deficiência",  "Discurso Social", r"pessoas? com deficiencia", "sim", "não"),
    ("deficiente físico",       "deficiente físico",       "Discurso Social", r"deficientes? fisic[oa]s?", "sim", "não"),
    ("portador de deficiência", "portador de deficiência", "Discurso Social", r"portador(?:a|es|as)? de deficiencia", "sim", "não"),
    ("autista",                 "",                        "Discurso Social", r"\bautistas?\b", "a confirmar com a Janaina", "não"),
    ("deficiente",              "",                        "Discurso Social", r"\bdeficientes?\b", "a confirmar com a Janaina", "não"),
    ("deficiente mental",       "",                        "Discurso Social", r"deficientes? menta(?:l|is)", "a confirmar com a Janaina", "não"),
    ("em cadeira de rodas",     "",                        "Discurso Social", r"cadeiras? de rodas", "a confirmar com a Janaina", "não"),
    ("retardado",               "",                        "Só E (xingamento)", r"\bretardad[oa]s?\b", "não", "não"),
    ("mongol",                  "",                        "Só E (xingamento)", r"\bmongol(?:oide)?s?\b", "não", "não"),
]
termos = pd.DataFrame(TERMOS, columns=["termo", "subclass_v1", "grupo", "padrao_busca", "entra_no_v2", "subconjunto_metaforico"])
termos["data_congelamento"] = DATA_CONGELAMENTO
termos.to_csv(f"{ROOT}/lista_termos_congelada.csv", index=False)
print(termos.to_string(index=False))

                               termo                          subclass_v1             grupo                        padrao_busca               entra_no_v2 subconjunto_metaforico data_congelamento
  com transtorno do espectro autista   com Transtorno do Espectro Autista           Clínico      transtorno do espectro autista                       sim                    não        2026-10-12
com transtorno de déficit de atenção com Transtorno de Déficit de Atenção           Clínico         deficit de atencao|\btdah\b                       sim                    não        2026-10-12
              com paralisia cerebral               com Paralisia Cerebral           Clínico                  paralisia cerebral                       sim                    não        2026-10-12
              com deficiência visual               com Deficiência Visual           Clínico                  deficiencia visual                       sim                    não        2026-10-12
                  com per

## 5. Contagem por termo e por rótulo

Número de textos de cada corpus que contêm cada termo, separados por rótulo. Termos que não aparecem em um corpus ficam fora da tabela dele.

In [8]:
def sem_acento(s):
    s = unicodedata.normalize("NFD", str(s).lower())
    return "".join(c for c in s if unicodedata.category(c) != "Mn")

rows = []
for name, d in CORPORA.items():
    texto = d.text.map(sem_acento)
    for _, t in termos.iterrows():
        achou = texto.str.contains(t.padrao_busca, regex=True)
        for label, n in d.label[achou].value_counts().items():
            rows.append({"corpus": name, "termo": t.termo, "grupo": t.grupo, "rotulo": label, "textos": int(n)})
contagem = pd.DataFrame(rows)
contagem.to_csv(f"{OUT}/c1_contagem_termos_por_corpus.csv", index=False)

for name in CORPORA:
    c = contagem[contagem.corpus == name]
    if len(c):
        print("\n=====", name, "| total por rótulo:", CORPORA[name].label.value_counts().to_dict())
        print(c.pivot(index="termo", columns="rotulo", values="textos").reindex(termos.termo).dropna(how="all").fillna(0).astype(int))


===== harmful-text (PT) | total por rótulo: {'harmless': 28103, 'harmful': 28103}
rotulo                                harmful  harmless
termo                                                  
com transtorno de déficit de atenção        1         0
com paralisia cerebral                      0         1
com perda auditiva                          0         1
com síndrome de down                        2         2
cego                                      525         9
surdo                                     212         1
pessoa com deficiência                   1716      1349
autista                                     3         1
deficiente                                855         8
deficiente mental                         369         0
em cadeira de rodas                       804       242
retardado                                 119        26
mongol                                     20         3

===== OLID-BR | total por rótulo: {'ofensivo': 5936, 'não ofensivo': 1016}
r

## Leitura dos resultados

**Vazamento**
- O ToxiGuardrail-PT viu as partições de treino e de validação do ToLD-Br (cerca de 70% dos textos de cada uma aparecem no `harmful-text`) e também a maior parte dos tuítes de Fortuna et al. (2019), que não estão listados na ficha do `harmful-text`.
- O teste do ToLD-Br, o HateCheck-PT, o ToxSyn-PT e o HateBR não foram vistos por nenhum dos dois modelos (no máximo 2,5% de textos coincidentes).
- Cerca de 2% dos textos do ToLD-Br aparecem no OLID-BR, e 5% dos textos do OLID-BR aparecem no `harmful-text`.
- O TuPy-E reúne quatro corpora: ToLD-Br (leite et al.), HateBR (vargas et al.), Fortuna et al. e TuPy (oliveira et al.). Só a parte de Oliveira et al. é nova em relação aos corpora que já usamos e não vista pelos modelos.

**Termos**
- No treino do ToxiGuardrail-PT, "cego", "surdo", "deficiente" e "deficiente mental" aparecem quase só do lado `harmful`. "Pessoa com deficiência" aparece dos dois lados em proporção parecida. "Cadeirante" não aparece.
- No OLID-BR, o treino do BERTimbau toxic, os termos do BITS são raros. Os mais frequentes são os xingamentos "retardado" e "mongol", quase sempre em textos ofensivos.
- No HateBR e no ToLD-Br, os termos do BITS aparecem em poucas dezenas de textos, o que confirma que esses corpora servem para calibração geral e para estudo de casos, não para estimar taxas por termo.